# 00 · The world

Chapter 00 takes no run and returns the toy case's inputs: an accounting structure built from its spec, three parameter tables, and a first look at the world that the three sources will describe, all registered in `runs/input/`.

## The story

**Orium** is a fictional metal. Between 2001 and 2005 it moves through three regions:

- **Montara** mines about 1,000 kt of ore a year, mills it, and exports most of the concentrate. Its own smelter is small.
- **Portua** is the port. It has smelters that run on imported concentrate and on scrap, and no mines at all.
- **Isla** is self-sufficient and modest in everything.

The chain is the one every metal has: ore is mined, milled into a concentrate, smelted into refined metal, and the metal is used or goes into stock. We track every flow in two **valuations**: `Gross` (the mass of the material, in `kt`) and `Content` (the orium inside it, in `kt Content`).

The world is closed: there is no rest-of-world region (`ROW`). Copper's account has one; the toy does not need it. The toy also declares its own labels (`Gross` and `Content`, where copper says `Mass` and `CC`), and has no hard bands. Each chapter says in one sentence where this matters.

This chapter sets the scene. The six stages start in chapter 01.

In [1]:
import json
import logging
from pathlib import Path

import pandas as pd

from pysuto.tutorial import toy

RUNS = Path("runs")
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 70)
logging.disable(logging.INFO)

## The accounting structure

Everything in pySUTO hangs on one object: the **accounting structure**. It lists the sectors, the regions, the years and the valuations, and from them every edge that may carry a flow. It is built from a small spec, a plain Python dictionary. The toy's is `toy.ACC_SPEC`; here are its thirteen sectors.

An **entity** says what kind of node a sector is: a `Process` transforms material, a `Flow` is a commodity that processes supply and use, and an `SB` (system boundary) is where material enters or leaves the account. An SB carries a `role` that tells the framework what it means, here nature, a secondary source or sink, or final demand.

In [2]:
sectors = pd.DataFrame(toy.ACC_SPEC["sectors"]).fillna("")
sectors

,name,entity,sub,role
0,Deposit,SB,Raw,nature
1,Mining,Process,,
2,Ore,Flow,,
3,WasteRock,SB,Sec,secondary
4,Milling,Process,,
5,Concentrate,Flow,,
6,Tailings,SB,Sec,secondary
7,Smelting,Process,,
8,Scrap,SB,Sec,secondary
9,Metal,Flow,,


Who makes what, and who takes what? That is the `supply` and `use` part of the spec: each process supplies commodities, and each commodity (or boundary) is used by processes.

In [3]:
spec = toy.ACC_SPEC
pd.DataFrame({"supplies": {p: ", ".join(v) for p, v in spec["supply"].items()}}).rename_axis("process")

,supplies
process,
Mining,"Ore, WasteRock"
Milling,"Concentrate, Tailings"
Smelting,"Metal, Slag"


In [4]:
pd.DataFrame({"is used by": {c: ", ".join(v) for c, v in spec["use"].items()}}).rename_axis("commodity or boundary")

,is used by
commodity or boundary,
Deposit,Mining
Ore,Milling
Concentrate,Smelting
Scrap,Smelting
Metal,"Use, Stock"


Read the two tables as a chain: `Deposit → Mining → Ore → Milling → Concentrate → Smelting → Metal → {Use, Stock}`. Three side flows leave the chain: `WasteRock` (from Mining), `Tailings` (from Milling) and `Slag` (from Smelting). `Scrap` re-enters at Smelting. `Concentrate` is the only commodity that crosses regions (`tradable`), and `Metal → Stock` is the one **signed** leg: a stock can shrink as well as grow.

Waste rock is a **satellite** flow: it has a value, but it sits outside Mining's mass balance (ADR 0038). The spec says so with a mask, which chapter 03 uses.

## Registering the inputs

The spec, and the parameters we meet next, are the case's *inputs*. `toy.write_inputs` lays them out under `runs/input/` the way a real data directory holds them (the spec as `acc/toy/current/acc.json`, each parameter table as `params/<id>/current/params.csv`) and registers them. The registry stamps every input with a version and a hash, so that a later run can say exactly which inputs it read.

In [5]:
inputs = toy.write_inputs(RUNS)
entry = inputs.acc.entry("toy")
print("acc entry:", {k: entry[k] for k in ("id", "version", "code")})

acc entry: {'id': 'toy', 'version': 1, 'code': '3e661eae'}


Now build the structure from the registered entry, as Stage 1 will do. The report lists how many node pairs, times and valuations the rules cut down to the admissible edges.

In [6]:
acc = toy.build_acc(RUNS / "proc" / "acc", entry)
per_year = acc.edge_meta.groupby(["time", "valuation"]).size()
print(f"{acc.n_edges} edges = {per_year.iloc[0]} per year and valuation × "
      f"{len(spec['times'])} years × {len(spec['valuations'])} valuations")

420 edges = 42 per year and valuation × 5 years × 2 valuations


Only these edges can carry a flow. The `accounting` rules of the spec forbid every other pair of nodes (a Process may supply a Flow only in its own region, a Flow may go to a Process in any region, and so on). A value outside this edge set cannot be stored, which is how the framework keeps the account well-formed.

<details><summary>Going deeper: the five accounting rules and the bounds</summary>

The `accounting` list of the spec is a whitelist of `(from-entity, to-entity, same region?)` triples. An edge exists only when its endpoints match one of them, and when `supply` or `use` allows the pair of sectors. That is what makes a Process → Flow edge a *supply* edge and a Flow → Process edge a *use* edge.

Every edge also has a lower and an upper bound, by default `[0, 1e9]`: a flow is never negative. The one exception is the signed leg, whose bounds the spec widens to `±1,000,000`. The toy registers no hard-band builder, so its solver does not enforce these bounds; copper's does (ADR 0039).
</details>

## The parameters

The spec fixes the *shape* of the account. Three small tables fix the *recipes*, and each row carries its value and its source:

- `conc` (**concentration**): how much orium one `kt` of a material holds, for example the grade of the ore or of the concentrate.
- `eff` (**efficiency**): how much of the orium that enters a process leaves it as the process's product, for example the milling recovery.
- `stripping_ratio`: how much waste rock is dug per `kt` of ore.

These are the parameters Stage 3 turns into balance rows. Here are the `conc` rows.

In [7]:
def params(param_id):
    path = RUNS / "input" / "params" / param_id / "current" / "params.csv"
    return pd.read_csv(path, sep=None, engine="python", dtype=str).fillna("")

params("conc")[["param_id", "description", "value_nominal", "unc_type", "value_sd"]]

,param_id,description,value_nominal,unc_type,value_sd
0,concoreusemontara,"orium content of Ore (Flow) on the edges into it, Montara",0.01,sd,0.001
1,concoresupplymontara,"orium content of Ore (Flow) on the edges out of it, Montara",0.01,sd,0.001
2,concoreuseisla,"orium content of Ore (Flow) on the edges into it, Isla",0.008,sd,0.0008
3,concoresupplyisla,"orium content of Ore (Flow) on the edges out of it, Isla",0.008,sd,0.0008
4,concconcentrateuse,"orium content of Concentrate (Flow) on the edges into it, every re...",0.28,sd,0.02
5,concconcentratesupply,"orium content of Concentrate (Flow) on the edges out of it, every ...",0.28,sd,0.02
6,concmetaluse,"orium content of Metal (Flow) on the edges into it, every region",0.99,sd,0.005
7,concmetalsupply,"orium content of Metal (Flow) on the edges out of it, every region",0.99,sd,0.005
8,concscrapsupply,"orium content of Scrap (SB) on the edges out of it, every region",0.7,sd,0.05
9,concslaguse,"orium content of Slag (SB) on the edges into it, every region",0.015,sd,0.003


The ore grades are about 1 % (`0.01` and `0.008`), the concentrate about 28 %, the metal 99 %. The last row sets the grade of waste rock to exactly zero. And now the other two tables.

In [8]:
pd.concat([params("eff"), params("stripping_ratio")])[
    ["param_id", "description", "value_nominal", "unc_type", "value_min", "value_max"]]

,param_id,description,value_nominal,unc_type,value_min,value_max
0,effmilling,"orium recovery of Milling: Ore in, Concentrate out",0.9,min_max,0.85,0.95
1,effsmelting,"orium recovery of Smelting: * in, Metal out (pooled inputs, scrap ...",0.97,min_max,0.95,0.99
0,strippingratio,"waste rock per ore mass at Mining, a satellite outside the Mining ...",2.0,min_max,1.5,2.5


Milling recovers `0.9` of the orium and smelting `0.97`; each is stated with a plausible range. The stripping ratio is `2.0` with a range of 1.5 to 2.5. Note the `agg=true` that the smelting row carries in the file: it pools every input of Smelting, concentrate and scrap together, into one recovery. Stage 3 uses it.

Every value has a `source`. Read it: these are synthetic numbers, and the world below was generated from them.

In [9]:
params("conc")["source"].iloc[0]

'pysuto.tutorial.toy: a synthetic case; its ground truth uses this nominal value'

<details><summary>Going deeper: anatomy of a parameter row</summary>

A parameter is a relation between two sets of edges, an A side and a B side, each given by the same columns as an edge (`sector_from`, `region_from`, `sector_to`, `time`, `valuation`, …; `*` is a wildcard, `same` copies the A side's value). A `conc` row relates the Gross value of the edges *into* or *out of* a sector (A) to the Content value of the same edges (B); an `eff` row relates the Content that enters a process (A) to the Content of its product (B).

`unc_type` says how the stated uncertainty is read: a standard deviation (`sd`) or a plausible range (`min_max`). Pooling over several inputs happens only where the row says `agg=true`, never by inference (ADR 0005, ADR 0022). Parameter ids contain no underscore, because Stage 3 builds its row ids from them.
</details>

## The truth, at world level

The toy has a **ground truth**: a perfectly balanced set of flows from which the feeds are generated. That is what no real case has. We keep it sealed until chapter 07, so that the pipeline cannot cheat and neither can you.

What we may look at is its **world totals**: every flow summed over the three regions. A sum over regions tells you the size of the world's metabolism, and nothing about who does what. Here are the world's main flows in `kt Content`, year by year.

In [10]:
wt = toy.world_totals()
main = [("Mining", "Ore"), ("Milling", "Concentrate"), ("Scrap", "Smelting"),
        ("Smelting", "Metal"), ("Metal", "Use"), ("Metal", "Stock")]
content = wt[wt["valuation"] == toy.CONTENT].set_index(["from_sector", "to_sector", "time"])["value"]
content.unstack("time").loc[main].round(2)

time                      2001   2002   2003   2004   2005
from_sector to_sector                                     
Mining      Ore          12.44  12.88  13.30  13.70  13.73
Milling     Concentrate  11.20  11.59  11.97  12.33  12.36
Scrap       Smelting      4.05   4.12   4.33   4.41   4.62
Smelting    Metal        14.80  15.24  15.81  16.24  16.47
Metal       Use          13.80  14.69  15.71  16.59  17.27
            Stock         1.00   0.55   0.10  -0.35  -0.80

The world's ore, concentrate and metal move with a slow trend; the stock leg is tiny and changes sign. Gross masses are far larger than their contents: orium is 1 % of the ore.

## Exercise

Every `Process` and `Flow` node of the world balances: what goes in comes out (Mining's waste rock aside, see below). Here are all flows of 2003, in `kt Content`.

In [11]:
year = content.xs("2003", level="time")
year.round(3)

from_sector  to_sector  
Deposit      Mining         13.303
Mining       Ore            13.303
             WasteRock       0.000
Ore          Milling        13.303
Milling      Concentrate    11.973
             Tailings        1.330
Concentrate  Smelting       11.973
Scrap        Smelting        4.330
Smelting     Metal          15.814
             Slag            0.489
Metal        Stock           0.100
             Use            15.714
Name: value, dtype: float64

**Exercise 0.1** — Close the world balance of `Smelting` in 2003: add up its inflows and its outflows from the table above and check the two sums with `toy.check_0_1`. Cover the next cell and try it first; `toy.solution("0.1")` prints the reference.

The check works at world level on purpose. A region's balance would need the region's numbers, and those would unseal the truth before chapter 07.

In [12]:
inflow = year[("Concentrate", "Smelting")] + year[("Scrap", "Smelting")]
outflow = year[("Smelting", "Metal")] + year[("Smelting", "Slag")]
print(toy.check_0_1(inflow, outflow, "Smelting", "2003"))

✓ 0.1  Smelting, world, 2003: in 16.3029 = out 16.3029 kt Content


The balance closes. Try `Milling` or `Metal` as well. `Mining` is the exception: its waste rock is a satellite outside the balance, and the exercise rejects it.

## Three sources

Nobody observes the truth directly. Three agencies publish numbers about it, each in its own vocabulary, and each with an error that fits its quality. We call them the **synthetic feeds**; chapter 01 reads them.

| Source | What it reports | Valuation | Names its regions by |
|---|---|---|---|
| **Survey** (a geological survey) | ore extraction of the mining regions | `Gross` | name |
| **Bureau** (a statistics bureau) | the chain in orium content: mine, concentrate, refined output (primary and secondary as two series), scrap, consumption, stock change; and Isla's ore tonnage | `Content`, and `Gross` for the ore | name |
| **Customs** (a customs authority) | bilateral concentrate trade | `Gross` | partner code |

Their quality differs, and so do their gaps. The Survey sees one flow in two regions. The Bureau sees the whole chain, though its tables have holes. Customs sees every border crossing, but only crossings. Chapter 01 puts a number on each source's quality.

> **One of the three carries one deliberate error. Find it before chapter 07.** The pipeline will not tell you directly; it will only tell you where the data disagree.

## Where this run came from

This chapter produced no run, so there is no run sidecar to read. It left two records instead. The input registry says what is registered and under which hash; the accounting structure's build record says which input entry it was built from.

In [13]:
registry = json.loads((RUNS / "input" / "acc" / "registry.json").read_text())
version = registry["toy"]["versions"][-1]
print("input registry :", {"version": version["version"], "code": version["code"],
                           "acc.json sha256": version["hashes"]["acc.json"][:12] + "…"})
print("build record   :", json.loads((RUNS / "proc" / "acc" / "toy" / "build_record.json").read_text()))

input registry : {'version': 1, 'code': '3e661eae', 'acc.json sha256': '9b184f992123…'}
build record   : {'id': 'toy', 'version': 1, 'code': '3e661eae'}


In [14]:
print(sorted(p.name for p in (RUNS / "input" / "params").iterdir() if p.is_dir()))

['conc', 'eff', 'stripping_ratio']


The two records agree: the structure was built from version 1 of the registered spec. Every later run records this entry in `parents.inputs`. Next: [chapter 01](01_datafeed.ipynb), where the three sources enter the pipeline.